# 📝 Cypher 심화 과제 LV3(통합): SNS 분석·경로 진단

> 이 단원의 문법을 묶어 **작은 분석 프로그램**을 만듭니다. 각 단계 셀의 지시를 따라 함수를 완성하고, 결과를 `output/` 에 리포트로 저장합니다.

## 풀이 방법
1. 맨 위 **준비 셀 → 초기화 셀 → 시드 적재 셀**을 순서대로 실행하세요.
2. 각 **단계 셀**의 지시대로 함수를 채우고, **자가채점 셀**로 확인하세요.

**도메인**: **SNS 친구 네트워크**입니다. `(A)-[:FRIEND]-(B)` 는 A 와 B 가 친구라는 뜻으로, **방향이 없습니다**(조회할 때 화살표 없이 `-[:FRIEND]-`). 각 사람(`Person`)은 `city`(사는 도시) 속성을 가집니다.

화이팅!

아래 세 셀(연결 → 초기화 → 시드 적재)을 먼저 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 앞 단원에서 만든 그래프(day28 의 Movies 예제, day29 의 과제 결과)도 함께 사라집니다. 되돌릴 수 없으니 `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요.

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙은 관계까지 함께 지우라는 뜻입니다.
run_cypher("MATCH (n) DETACH DELETE n")
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

오늘 만들 프로그램이 다룰 그래프입니다. 사람 8명이 친구 관계 9개로 이어져 있습니다.

<img src="images/친구망_그래프.png" width="820">

In [ ]:
# [제공 코드] SNS 친구 네트워크 시드 적재: 이 셀은 실행만 하세요(그래프를 처음부터 만듭니다).
run_cypher("""
CREATE (ari:Person {name:'아리', city:'서울'}),
       (bomi:Person {name:'보미', city:'서울'}),
       (chris:Person {name:'크리스', city:'부산'}),
       (dana:Person {name:'다나', city:'서울'}),
       (eun:Person {name:'은수', city:'대구'}),
       (fin:Person {name:'피니', city:'부산'}),
       (gale:Person {name:'가을', city:'서울'}),
       (dal:Person {name:'달이', city:'제주'})
CREATE (ari)-[:FRIEND]->(bomi),
       (ari)-[:FRIEND]->(chris),
       (bomi)-[:FRIEND]->(dana),
       (chris)-[:FRIEND]->(eun),
       (dana)-[:FRIEND]->(fin),
       (eun)-[:FRIEND]->(gale),
       (fin)-[:FRIEND]->(gale),
       (bomi)-[:FRIEND]->(chris),
       (dana)-[:FRIEND]->(chris)
""")
print("SNS 적재 완료. 사람:", len(run_cypher("MATCH (p:Person) RETURN p.name")), "명")


## 데이터 살펴보기
아래 셀은 **실행만** 하세요. 누가 누구와 친구인지 훑어봅니다.

In [ ]:
# [제공 코드] 친구 관계를 먼저 훑어봅니다(실행만 하세요)
for r in run_cypher("MATCH (a:Person)-[:FRIEND]->(b:Person) "
                    "RETURN a.name AS 사람, b.name AS 친구 ORDER BY 사람, 친구"):
    print(r['사람'], '·', r['친구'])

---
## 1. 친구 추천 프로그램
**배경**: SNS 는 "알 수도 있는 사람"을 추천합니다. **친구의 친구**(2홉) 중 아직 내 친구가 아닌 사람을 후보로 뽑는 미니 프로그램을 단계별로 만듭니다.

아래 **세 단계**(1-1 ~ 1-3)를 순서대로 완성하세요. 각 단계 셀에 그 단계의 할 일이 적혀 있습니다.

### 1-1. `friends(name)`: 직접 친구 목록
사람 이름을 받아 **직접 친구**(1홉) 이름을 **정렬된 리스트**로 돌려주는 함수 `friends(name)` 를 만드세요.

- `FRIEND` 를 **방향 없이**(`-[:FRIEND]-`) 잇고, 파라미터 `$name` 을 씁니다. 친구 이름을 반환 컬럼 별칭 `n` 으로.
- 파이썬에서 `sorted(...)` 로 정렬해 돌려줍니다.

**예시**: `friends('아리')` → `['보미', '크리스']`, `friends('크리스')` → `['다나', '보미', '아리', '은수']`.

<details><summary>힌트</summary>

```text
접근방법:
- FRIEND 를 방향 없이 이어 친구 이름을 모으고 파이썬에서 정렬한다.

세부구현:
1. run_cypher 로, 이름이 name 인 사람에서 FRIEND 를 방향 없이 이은 상대의 이름을 별칭 n 으로 받는다(파라미터 name=name).
2. 결과에서 이름만 모아 sorted 로 정렬해 return 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert friends('아리') == ['보미', '크리스'], \
    'FRIEND 를 방향 없이 이었는지, 파라미터 $name 을 썼는지, 파이썬에서 sorted 로 정렬해 돌려줬는지 확인하세요'
assert friends('크리스') == ['다나', '보미', '아리', '은수'], \
    'FRIEND 를 방향 없이 이었는지, 파라미터 $name 을 썼는지, 파이썬에서 sorted 로 정렬해 돌려줬는지 확인하세요'
print('✅ 통과!')

### 1-2. `recommend(name)`: 2홉 추천 후보
**친구의 친구**(2홉) 중 **본인**과 **이미 친구인 사람**을 빼고 남는 사람을 추천 후보로 돌려주는 함수 `recommend(name)` 를 만드세요.

- Cypher 로 **2홉 이웃**(친구의 친구)을 모읍니다: `FRIEND` 를 두 번 이어 `fof` 를 얻습니다.
- `WHERE` 에서 **두 가지를 함께** 거릅니다. **본인 제외**(`fof <> a`)와 **이미 친구인 사람 제외**(`NOT (a)-[:FRIEND]-(fof)`). 뒤엣것이 교안_02 2-2 의 **패턴 술어**입니다.
- 그다음 **`WITH DISTINCT`** 로 중복을 없애 이름을 별칭 `n` 으로 반환합니다.
- 파이썬으로 후처리하지 말고 **Cypher 안에서 다 거른 뒤** 이름만 받아 `sorted` 로 정렬해 돌려줍니다.

**예시**: `recommend('아리')` → `['다나', '은수']`(다나=보미의 친구, 은수=크리스의 친구), `recommend('보미')` → `['은수', '피니']`.

<details><summary>힌트</summary>

```text
접근방법:
- 2홉 이웃을 모으고, '본인' 과 '이미 친구' 를 둘 다 WHERE 에서 걸러 낸다.

세부구현:
1. run_cypher 로, 이름이 name 인 사람 a 에서 FRIEND 를 두 번 이어(친구의 친구) fof 를 얻는다.
2. WHERE 에 fof <> a 로 본인을 빼고, NOT (a)-[:FRIEND]-(fof) 로 이미 친구인 사람을 뺀다.
   패턴 술어에는 새 변수를 만들 수 없으니 이미 있는 a·fof 를 그대로 쓴다.
3. WITH DISTINCT 로 fof 이름을 별칭 n 으로 넘겨 반환하고, 파이썬에서는 sorted 로 정렬만 해 return 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert recommend('아리') == ['다나', '은수'], \
    'FRIEND 를 두 번 이어 2홉 이웃을 모았는지, WHERE 에서 본인(fof <> a)과 이미 친구(NOT (a)-[:FRIEND]-(fof))를 둘 다 뺐는지, WITH DISTINCT 를 썼는지 확인하세요'
assert recommend('보미') == ['은수', '피니'], \
    'FRIEND 를 두 번 이어 2홉 이웃을 모았는지, WHERE 에서 본인(fof <> a)과 이미 친구(NOT (a)-[:FRIEND]-(fof))를 둘 다 뺐는지, WITH DISTINCT 를 썼는지 확인하세요'
print('✅ 통과!')

### 1-3. 추천 리포트 저장
`['아리', '보미', '크리스']` 세 사람 각각에 대해 `friends` 와 `recommend` 결과를 담은 리포트 사전을 만들어 **`output/sns_recommend.json`** 으로 저장하세요.

- 리포트 구조: **바깥 사전은 사람 이름이 열쇠**이고, 그 값은 `'friends'`·`'recommend'` 두 열쇠를 가진 사전입니다: `{ 이름: {'friends': [...], 'recommend': [...]}, ... }`
- 변수 이름은 **`report`**. 저장할 폴더는 아직 없으니 **먼저 만들어야 합니다**(`os.makedirs(..., exist_ok=True)`). 파일은 `encoding='utf-8'` 로 열고 `json.dump(report, f, ensure_ascii=False, indent=2)` 로 씁니다.

**저장될 파일의 모습**(아리 부분만 채워 보인 것):

```json
{
  "아리": {"friends": ["보미", "크리스"], "recommend": ["다나", "은수"]},
  "보미": {"friends": [...], "recommend": [...]},
  "크리스": {"friends": [...], "recommend": [...]}
}
```

**예시**: `report['아리']['recommend']` 는 `recommend('아리')` 와 같아야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 세 사람을 돌며 friends·recommend 를 호출해 사전에 모으고 json 으로 저장한다.

세부구현:
1. json 과 os 를 불러오고, 저장할 output 폴더가 없으면 만든다(os 의 폴더 생성 함수를 쓰되
   이미 있어도 오류가 나지 않는 옵션을 준다).
2. 빈 사전으로 시작해 세 사람을 차례로 돌며, 그 사람 이름을 열쇠로 두고
   값 자리에 friends 결과와 recommend 결과를 각각 지문에 적힌 열쇠 이름으로 담는다.
3. 지정된 경로를 쓰기 모드·utf-8 로 열고 json.dump 로 저장한다(지문에 적힌 옵션 그대로).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import json
saved = json.load(open('output/sns_recommend.json', encoding='utf-8'))
assert set(saved) == {'아리', '보미', '크리스'}, \
    '세 사람 이름을 열쇠로 담았는지 확인하세요'
assert saved['아리']['friends'] == ['보미', '크리스'], \
    "각 사람 아래에 'friends' 열쇠로 friends 결과를 담았는지 확인하세요"
assert saved['아리']['recommend'] == ['다나', '은수'], \
    "각 사람 아래에 'recommend' 열쇠로 recommend 결과를 담았는지 확인하세요"
assert saved['보미']['recommend'] == ['은수', '피니'], \
    '세 사람 모두를 돌았는지 확인하세요(한 사람만 담고 끝내지 않았는지)'
print('✅ 통과!')

---
## 2. 경로 진단 리포트
**배경**: 두 사람이 **어떻게 연결되는지**(몇 다리 건너, 누구를 거쳐), 혹은 **아예 연결이 안 되는지**를 진단하는 리포트를 만듭니다.

아래 **두 단계**(2-1 ~ 2-2)를 순서대로 완성하세요.

### 2-1. `diagnose(a, b)`: 두 사람 연결 진단
두 사람 이름을 받아 다음을 담은 **사전**을 돌려주는 함수 `diagnose(a, b)` 를 만드세요.

- `connected`: 연결 여부(참/거짓): 경로가 `null` 이 아닌지(`p IS NOT NULL`)
- `hops`: 최단 연결의 다리 수(`length(p)`). 연결 안 되면 `None`
- `path`: 거쳐가는 사람 이름 목록(순서대로, `nodes(p)` 에서 이름만). 연결 안 되면 `None`
- `all_seoul`: 경로가 지나는 사람이 **전부 서울** 사람인지(`all(n IN nodes(p) WHERE n.city = '서울')`). 연결 안 되면 `None`

친구 관계는 **방향이 없습니다**(`-[:FRIEND*]-`). 그래서 `diagnose('아리', '가을')` 과 `diagnose('가을', '아리')` 는 **같은 길이·뒤집힌 경로**가 나와야 합니다. 화살표를 붙이면 한쪽 순서에서만 답이 나옵니다.

두 사람을 이름으로 잡고 `OPTIONAL MATCH` + `shortestPath` 로 `FRIEND` 경로를 구하세요. 경로가 없으면 `p` 가 `null` 이 되고, `length(p)`·`nodes(p)` 도 자연히 `null` 이 되므로 **따로 분기하지 않아도** `hops`·`path` 가 알아서 `None` 이 됩니다. 반환 컬럼 별칭은 `connected`·`hops`·`path`·`all_seoul` 네 가지이고, 함수는 그 첫 행(dict)을 그대로 돌려주면 됩니다.

**예시**: `diagnose('아리', '가을')` → `connected=True, hops=3, path=['아리', '크리스', '은수', '가을'], all_seoul=False`. `diagnose('아리', '달이')` → `connected=False` 이고 나머지 셋은 모두 `None` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- OPTIONAL MATCH + shortestPath 로 경로 p 를 구하고, p 가 없으면 length·nodes 가 저절로 null 이 됨을 이용한다.

세부구현:
1. 두 사람을 각각 이름으로 잡는다.
2. OPTIONAL MATCH 로 둘 사이 shortestPath(FRIEND 를 방향 없이 가변길이로)를 구해 경로 p 에 담는다.
3. p IS NOT NULL 을 connected, length(p) 를 hops, nodes(p) 의 이름 목록을 path 로 반환한다(별도 분기 불필요).
4. all(n IN nodes(p) WHERE n.city = '서울') 을 all_seoul 로 함께 반환한다.
5. 결과의 첫 행(rows[0])을 return 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
r1 = diagnose('아리', '가을')
assert r1['connected'] == True and r1['hops'] == 3 and r1['path'] == ['아리', '크리스', '은수', '가을'], \
    'connected·hops·path 별칭을 다 반환했는지, FRIEND 를 방향 없이 이었는지, 첫 행(dict)을 그대로 돌려줬는지 확인하세요'
assert r1['all_seoul'] == False, \
    "all(n IN nodes(p) WHERE n.city = '서울') 을 all_seoul 별칭으로 함께 반환했는지 확인하세요"
r3 = diagnose('가을', '아리')
assert r3['connected'] == True and r3['hops'] == 3 and r3['path'] == ['가을', '은수', '크리스', '아리'], \
    '친구 관계는 방향이 없습니다. -[:FRIEND*]- 로 이어야 순서를 뒤집어도 같은 길이의 경로가 나옵니다'
r2 = diagnose('아리', '달이')
assert r2['connected'] == False and r2['hops'] is None and r2['path'] is None and r2['all_seoul'] is None, \
    '경로가 없을 때 hops·path·all_seoul 이 None 이 되게 두었는지 확인하세요(따로 분기하면 오히려 어긋납니다)'
print('✅ 통과!')

### 2-2. 진단 리포트 저장
짝 목록 `[('아리','가을'), ('아리','피니'), ('아리','달이')]` 각각을 `diagnose` 로 진단해 **`output/sns_path_report.json`** 으로 저장하세요.

- 리포트는 **리스트**이고 짝 하나가 사전 하나입니다: `[{'a': ..., 'b': ..., 'connected': ..., 'hops': ..., 'path': ...}, ...]`
- 각 사전의 열쇠는 위 **다섯 개뿐입니다**. `diagnose` 가 함께 돌려주는 `all_seoul` 은 이 리포트에 **담지 않습니다**. 즉 `diagnose` 결과를 통째로 합치지 말고 `connected`·`hops`·`path` 셋만 꺼내 쓰세요.
- 변수 이름은 **`path_report`**. 저장 폴더가 없으면 먼저 만들고, 파일은 `encoding='utf-8'` 로 열어 `json.dump(path_report, f, ensure_ascii=False, indent=2)` 로 씁니다.
- 그리고 아래 **서술 답안 셀**에, 리포트에서 `connected=False` 로 나온 짝을 보고 **그 사람이 왜 아무와도 이어지지 않는지**와, **그런 사람을 매일 미리 찾아내려면 어떤 쿼리를 돌리면 될지**를 두세 문장으로 적으세요.

**저장될 파일의 모습**(가운데 짝은 비워 둔 것):

```json
[
  {"a": "아리", "b": "가을", "connected": true, "hops": 3, "path": ["아리", "크리스", "은수", "가을"]},
  {"a": "아리", "b": "피니", "connected": true, "hops": 3, "path": [...]},
  {"a": "아리", "b": "달이", "connected": false, "hops": null, "path": null}
]
```

가운데 짝(`아리`와 `피니`)은 같은 길이의 최단 경로가 두 갈래라 `path` 가 둘 중 어느 쪽으로 나와도 맞습니다. 채점도 이 짝의 `path` 는 값으로 견주지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 짝을 돌며 diagnose 결과에서 필요한 셋만 꺼내 a·b 와 함께 리스트에 모으고 json 으로 저장한다.

세부구현:
1. 지문에 적힌 세 짝을 튜플 리스트로 준비한다.
2. 짝을 하나씩 돌며 diagnose 를 부르고, 그 결과에서 connected·hops·path 만 꺼내
   두 사람 이름(a·b)과 함께 담아 지문의 다섯 열쇠짜리 사전 하나를 만들어 리스트에 쌓는다.
   diagnose 결과를 통째로 합치면 all_seoul 까지 섞여 열쇠가 여섯 개가 된다(채점에서 걸린다).
3. 지정된 경로에 json.dump 로 저장한다(지문에 적힌 옵션 그대로).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import json
saved = json.load(open('output/sns_path_report.json', encoding='utf-8'))
assert len(saved) == 3, '세 짝을 모두 진단해 담았는지 확인하세요'
assert all(set(row) == {'a', 'b', 'connected', 'hops', 'path'} for row in saved), \
    '각 항목이 a·b·connected·hops·path 딱 다섯 열쇠만 갖는지 확인하세요(all_seoul 은 이 리포트에 담지 않습니다)'
gale = [r for r in saved if r['b'] == '가을'][0]
assert gale['connected'] == True and gale['path'] == ['아리', '크리스', '은수', '가을'], \
    'diagnose 결과를 그대로 담았는지 확인하세요'
dal = [r for r in saved if r['b'] == '달이'][0]
assert dal['connected'] == False and dal['path'] is None, \
    '연결이 없는 짝도 빠뜨리지 않고 담았는지 확인하세요'
print('✅ 통과!')

**서술 답안** *(리포트를 보고 위 두 가지를 적으세요. 정답 노트북의 모범답안과 비교)*

*(여기에 서술)*

---
수고했어요! 경로 탐색·다중 조건·**패턴 술어**·`OPTIONAL MATCH`·`WITH` 파이프라인을 묶어 추천과 경로 진단 프로그램을 완성했습니다. 특히 추천에서는 "이미 친구인 사람 빼기"를 파이썬이 아니라 **Cypher 안에서** 끝냈습니다. 다음 단원에서는 여러 행을 **하나로 요약**하는 집계와 **인덱스**로 나아갑니다.